In [ ]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 40.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=c06e7b3eb2c0948e3917d3798927639c4a587ad8f7c70dd811829f8c074fb7e0
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [ ]:
import chess.pgn

import chess.polyglot

import zlib

from collections import defaultdict, Counter

import io

import zstandard as zstd
import json

In [ ]:
import pickle


def process_single_file(file_path):
    transitions = defaultdict(lambda: Counter())
    outcomes = defaultdict(lambda: [0, 0, 0])
    node_occurrence = Counter()
    hash_to_fen = {}

    dctx = zstd.ZstdDecompressor()
    with open(file_path, "rb") as fh:
        with dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8')
            while True:
                game = chess.pgn.read_game(text_stream)
                if game is None: break

                 # --- FILTERS ---
                headers = game.headers
                try:
                    w_elo = int(headers.get("WhiteElo", 0))
                    b_elo = int(headers.get("BlackElo", 0))
                    time_ctrl = headers.get("TimeControl", "0+0")
                    main_time = int(time_ctrl.split('+')[0]) if '+' in time_ctrl else 0
                except (ValueError, IndexError):
                    continue

                if w_elo < 1000 or b_elo < 1000 or main_time < 300:
                    continue

                res = game.headers.get("Result", "*")
                res_idx = 0 if res == "1-0" else 1 if res == "0-1" else 2

                board = game.board()
                for move in game.mainline_moves():
                    curr_hash = chess.polyglot.zobrist_hash(board)
                    if curr_hash not in hash_to_fen:
                        hash_to_fen[curr_hash] = board.fen()

                    board.push(move)
                    next_hash = chess.polyglot.zobrist_hash(board)

                    transitions[curr_hash][next_hash] += 1
                    node_occurrence[curr_hash] += 1
                    outcomes[curr_hash][res_idx] += 1

    # Save intermediate results so we don't have to keep them in RAM
    output_name = file_path + ".tmp.pkl"
    with open(output_name, "wb") as f:
        pickle.dump((dict(transitions), dict(outcomes), dict(node_occurrence), hash_to_fen), f)
    return output_name

In [ ]:
def merge_and_prune(temp_files, max_nodes=1000, min_count=10, min_prob=0.02):
    master_transitions = defaultdict(lambda: Counter())
    master_outcomes = defaultdict(lambda: [0, 0, 0])
    master_occurrence = Counter()
    master_fens = {}

    for f_path in temp_files:
        print(f"Merging {f_path}...")
        with open(f_path, "rb") as f:
            trans, out, occ, fens = pickle.load(f)

            # Update FENs
            master_fens.update(fens)
            # Update Occurrences
            master_occurrence.update(occ)
            # Update Outcomes
            for h, vals in out.items():
                master_outcomes[h][0] += vals[0]
                master_outcomes[h][1] += vals[1]
                master_outcomes[h][2] += vals[2]
            # Update Transitions
            for h, targets in trans.items():
                master_transitions[h].update(targets)

    # --- NOW RUN PASS 2 (The Pruning) ---
    print("Pruning global data...")
    final_adj_list = {}
    top_nodes = {h for h, count in master_occurrence.most_common(max_nodes)}

    for curr_hash in top_nodes:
        total_exits = sum(master_transitions[curr_hash].values())
        if total_exits < min_count: continue

        valid_edges = {}
        for next_hash, count in master_transitions[curr_hash].items():
            prob = count / total_exits
            if prob >= min_prob and next_hash in top_nodes:
                valid_edges[str(next_hash)] = {"prob": round(prob, 4), "count": count}

        if valid_edges:
            final_adj_list[str(curr_hash)] = {
                "edges": valid_edges,
                "outcomes": master_outcomes[curr_hash],
                "total_visits": total_exits,
                "fen": master_fens.get(curr_hash)
            }
    return final_adj_list

In [ ]:
import os

# --- SETTINGS FOR THIS RUN ---
# Change this for each new file you download via !wget
MY_FILE_PATH = 'INSERT_FILE_NAME_HERE' #Should look something like this: '/content/lichess_db_standard_rated_2013-01 (3).pgn.zst'

# 1. Run the Phase 1 Parser
print(f"--- Starting Phase 1 for {MY_FILE_PATH} ---")
tmp_path = process_single_file(MY_FILE_PATH)

# 2. Cleanup: Delete the massive .zst to free up space for the next download
if os.path.exists(tmp_path):
    print(f"Success! Created {tmp_path}")
    # !rm {MY_FILE_PATH}  # Uncomment this to auto-delete the 30GB file after pickling

--- Phase 1: Parsing PGN to Temporary Pickle ---
Phase 1 Complete. Temporary file created at: /content/lichess_db_standard_rated_2013-01 (3).pgn.zst.tmp.pkl
--- Phase 2: Merging, Pruning, and Creating Final Graph ---
Merging /content/lichess_db_standard_rated_2013-01 (3).pgn.zst.tmp.pkl...
Pruning global data...
Saving final graph to chess_graph_FINAL.json...
--- All Done! ---
You can now use 'chess_graph_FINAL.json' to build your NetworkX graph.


In [ ]:
import glob

# 1. Find all temporary files created by Phase 1
all_temp_files = glob.glob("/content/*.tmp.pkl")
print(f"Found {len(all_temp_files)} files to merge: {all_temp_files}")

# 2. Run the Global Merge and Prune
# This is where max_nodes=1000 will finally be enforced across ALL data
final_data = merge_and_prune(all_temp_files, max_nodes=1000, min_count=10, min_prob=0.02)

# 3. Save the final result
output_name = "master_chess_graph.json"
with open(output_name, "w") as f:
    json.dump(final_data, f)

print(f"--- Finished! ---")
print(f"Combined {len(all_temp_files)} files into a {len(final_data)} node graph.")